In [2]:
import torch, os
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [16]:
!pip install --upgrade transformers==4.39.3
!pip install accelerate
!pip install open_clip_torch
!pip install torch-ema

  Using cached torch_ema-0.3-py3-none-any.whl.metadata (415 bytes)


In [3]:
import torch
torch.cuda.is_available()

True

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
ZIP_PATH = "/content/drive/MyDrive/data.zip"

In [6]:
!unzip -q "$ZIP_PATH" -d /content/data

In [7]:
TRAIN_DIR = "/content/data/data/train"
VAL_DIR   = "/content/data/data/val"

In [13]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import open_clip

model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    'ViT-H-14',
    pretrained='laion2b_s32b_b79k'
)

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=preprocess_train)
val_dataset   = datasets.ImageFolder(VAL_DIR,   transform=preprocess_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=8, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=8, pin_memory=True)

print("클래스:", train_dataset.classes)
print("Train imgs:", len(train_dataset))
print("Val imgs:", len(val_dataset))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/3.94G [00:00<?, ?B/s]

클래스: ['fake', 'real']
Train imgs: 72067
Val imgs: 18039


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import open_clip

class SRMConv(nn.Module):
    def __init__(self):
        super().__init__()
        kernel = torch.tensor([
            [-1,-1,-1],
            [-1, 8,-1],
            [-1,-1,-1]
        ], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        self.register_buffer("weight", kernel)

    def forward(self, x):
        gray = x.mean(dim=1, keepdim=True)
        return F.conv2d(gray, self.weight, padding=1)


class Deepfake_ViTH14_SRMBRANCH(nn.Module):
    def __init__(self):
        super().__init__()
        # ✅ 진짜 ViT-H/14 로딩 (정확한 모델 ID)
        self.backbone, self.preprocess, _ = open_clip.create_model_and_transforms(
            'ViT-H-14', pretrained='laion2b_s32b_b79k'
        )

        self.srm = SRMConv()
        self.fuse = nn.Conv2d(4, 3, kernel_size=1)

        hidden = self.backbone.visual.output_dim   # ViT-H = 1024 dim
        self.fc = nn.Linear(hidden, 2)

    def forward(self, x):
        srm = self.srm(x)
        fused = torch.cat([x, srm], dim=1)  # (B,4,H,W)
        fused = self.fuse(fused)            # (B,3,H,W)
        feats = self.backbone.encode_image(fused)
        return self.fc(feats)


# ----------------- LLRD (Layer-Wise LR Decay) -----------------
def get_layerwise_lr_params(model, base_lr=5e-6, decay=0.75):
    """
    model: Deepfake_ViTH14_SRMBRANCH()
    base_lr: 가장 하위 계층의 learning rate
    decay: layer depth마다 lr 감소 비율
    """
    layers = list(model.backbone.visual.transformer.resblocks)
    params = []

    # 각 레이어 깊이에 따라 learning rate 다르게 적용
    for i, layer in enumerate(layers):
        lr = base_lr * (decay ** (len(layers) - i))  # 깊을수록 lr 작게
        params.append({"params": layer.parameters(), "lr": lr})

    # patch embedding 부분
    params.append({"params": model.backbone.visual.conv1.parameters(), "lr": base_lr})
    params.append({"params": model.backbone.visual.class_embedding, "lr": base_lr})
    params.append({"params": model.backbone.visual.positional_embedding, "lr": base_lr})

    # SRM + fuse + FC (최종 classifier)는 lr 높게
    params.append({"params": model.srm.parameters(), "lr": 1e-4})
    params.append({"params": model.fuse.parameters(), "lr": 1e-4})
    params.append({"params": model.fc.parameters(),  "lr": 1e-4})

    return params

In [17]:
from torch.cuda.amp import GradScaler, autocast
from torch_ema import ExponentialMovingAverage
from sklearn.metrics import f1_score
from tqdm import tqdm

device = "cuda"
model = Deepfake_ViTH14_SRMBRANCH().to(device)

params = get_layerwise_lr_params(model)
optimizer = torch.optim.AdamW(params, weight_decay=0.05)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

ema = ExponentialMovingAverage(model.parameters(), decay=0.999)
scaler = GradScaler()

best_f1 = 0.0
save_path = "/content/drive/MyDrive/deepfake/model/ViT-H14_SRMBRANCH.pth"

for epoch in range(1, 9):
    model.train()
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch}"):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        with autocast():
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        ema.update()

    model.eval()
    preds, trues = [], []
    with ema.average_parameters(), torch.no_grad(), autocast():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            p = model(x).argmax(1)
            preds += p.cpu().tolist()
            trues += y.cpu().tolist()

    f1 = f1_score(trues, preds, average="weighted")
    print(f"[Epoch {epoch}] F1={f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), save_path)
        print(f"🔥 Best Updated → {save_path} (F1={f1:.4f})")


/tmp/ipython-input-59824407.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch 1:   0%|          | 0/2253 [00:00<?, ?it/s]/tmp/ipython-input-59824407.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1: 100%|██████████| 2253/2253 [12:50<00:00,  2.92it/s]
/tmp/ipython-input-59824407.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ema.average_parameters(), torch.no_grad(), autocast():


[Epoch 1] F1=0.9017
🔥 Best Updated → /content/drive/MyDrive/deepfake/model/ViT-H14_SRMBRANCH.pth (F1=0.9017)


Epoch 2:   0%|          | 0/2253 [00:00<?, ?it/s]/tmp/ipython-input-59824407.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 2: 100%|██████████| 2253/2253 [12:46<00:00,  2.94it/s]
/tmp/ipython-input-59824407.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ema.average_parameters(), torch.no_grad(), autocast():


[Epoch 2] F1=0.9055
🔥 Best Updated → /content/drive/MyDrive/deepfake/model/ViT-H14_SRMBRANCH.pth (F1=0.9055)


Epoch 3:   0%|          | 0/2253 [00:00<?, ?it/s]/tmp/ipython-input-59824407.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 3: 100%|██████████| 2253/2253 [12:46<00:00,  2.94it/s]
/tmp/ipython-input-59824407.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ema.average_parameters(), torch.no_grad(), autocast():


...

[Epoch 6] F1=0.9185
🔥 Best Updated → /content/drive/MyDrive/deepfake/model/ViT-H14_SRMBRANCH.pth (F1=0.9185)


Epoch 7:   0%|          | 0/2253 [00:00<?, ?it/s]/tmp/ipython-input-59824407.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 7: 100%|██████████| 2253/2253 [12:44<00:00,  2.95it/s]
/tmp/ipython-input-59824407.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ema.average_parameters(), torch.no_grad(), autocast():


[Epoch 7] F1=0.9198
🔥 Best Updated → /content/drive/MyDrive/deepfake/model/ViT-H14_SRMBRANCH.pth (F1=0.9198)


Epoch 8:   0%|          | 0/2253 [00:00<?, ?it/s]/tmp/ipython-input-59824407.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 8: 100%|██████████| 2253/2253 [12:44<00:00,  2.95it/s]
/tmp/ipython-input-59824407.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ema.average_parameters(), torch.no_grad(), autocast():


[Epoch 8] F1=0.9205
🔥 Best Updated → /content/drive/MyDrive/deepfake/model/ViT-H14_SRMBRANCH.pth (F1=0.9205)


In [18]:
import open_clip, pickle, os

_, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    'ViT-H-14', pretrained='laion2b_s32b_b79k'
)

save_dir = "/content/drive/MyDrive/deepfake/model/"
os.makedirs(save_dir, exist_ok=True)

with open(save_dir + "preprocess.pkl", "wb") as f:
    pickle.dump(preprocess_val, f)

print("✅ preprocess 저장 완료")

✅ preprocess 저장 완료


In [ ]:
import torch

src = "/content/drive/MyDrive/deepfake/model/ViT-H14_SRMBRANCH.pth"
dst = "/content/drive/MyDrive/deepfake/model/ViT-H14_SRMBRANCH_fp16.pth"

state = torch.load(src, map_location="cpu")

for k in list(state.keys()):
    v = state[k]
    if isinstance(v, torch.Tensor) and v.dtype == torch.float32:
        state[k] = v.half()

torch.save(state, dst)
print("✅ FP16 모델 저장 완료 →", dst)

✅ FP16 모델 저장 완료 → /content/drive/MyDrive/deepfake/model/ViT-H14_SRMBRANCH_fp16.pth
